In [3]:
import pandas as pd
import csv
import ast
import re


In [ ]:
with open('./qilin_rerank/raw_data/recommendation_train.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    # convert to pandas dataframe (first line is header)
    train_rec_df = pd.DataFrame(list(reader))
    train_rec_df.columns = train_rec_df.iloc[0]
    train_rec_df = train_rec_df.iloc[1:]
train_rec_df.head()

,recent_clicked_note_idxs,request_idx,session_idx,user_idx,query,rec_result_details_with_idx
1,[1607695 1636621 1407779 1434110 992023 9733...,23194,55877,6948,花火声优Cos花火 你醒啦？你刚刚好像做噩梦了 无语无语，太无语 来更新了，别催了 画了无数...,"[{'click': 1, 'collect': 0, 'comment': 0, 'lik..."
2,[1607695 1636621 1407779 1434110 992023 9733...,12121,55467,6948,花火声优Cos花火 你醒啦？你刚刚好像做噩梦了 无语无语，太无语 来更新了，别催了 画了无数...,"[{'click': 0, 'collect': 0, 'comment': 0, 'lik..."
3,[1607695 1636621 1407779 1434110 992023 9733...,44225,54091,6948,花火声优Cos花火 你醒啦？你刚刚好像做噩梦了 无语无语，太无语 来更新了，别催了 画了无数...,"[{'click': 1, 'collect': 0, 'comment': 0, 'lik..."
4,[1311477 1483454 1400183 847263 967940 9551...,41244,79569,11094,南京跨年音乐节来袭，即将官宣！ 优化真的好明显 吃几根脆脆烤牛肠 给绝育宠物装假体蛋蛋 南京...,"[{'click': 0, 'collect': 0, 'comment': 0, 'lik..."
5,[1518787 1372754 1613393 1514777 1530888 12853...,8596,79684,11897,推荐那些超搞笑的音乐🥬💕 推荐适合发癫的BGM🌝💕 “亲爱的 要和我喝一杯吗” 祝你生日快乐...,"[{'click': 1, 'collect': 0, 'comment': 0, 'lik..."


In [5]:
# filter the positive samples
train_rec_result = []
for idx, row in train_rec_df.iterrows():
    user_idx = row['user_idx']
    # 解析字符串为list
    rec_str = row['rec_result_details_with_idx']
    try:
        rec_list = re.findall(r"\{.*?\}", rec_str, re.DOTALL)
    except Exception as e:
        continue

    gt_note_list = []
    neg_note_list = []
    for rec in rec_list:
        rec_dict = ast.literal_eval(rec)
        if rec_dict.get('click', 0) == 1:
            note_idx = rec_dict.get('note_idx', None)
            timestamp = rec_dict.get('request_timestamp', None)
            gt_note_list.append(note_idx)
        else:
            note_idx = rec_dict.get('note_idx', None)
            timestamp = rec_dict.get('request_timestamp', None)
            neg_note_list.append(note_idx)
    history = row['recent_clicked_note_idxs']
    # change to list
    history = history.strip('[]').split()
    train_rec_result.append([user_idx, gt_note_list, neg_note_list, timestamp, history])

# 转为DataFrame
train_rec_result_df = pd.DataFrame(train_rec_result, columns=['user_idx', 'gt_note_idx', 'neg_note_idx', 'timestamp', 'history'])
train_rec_result_df.head()

,user_idx,gt_note_idx,neg_note_idx,timestamp,history
0,6948,"[885037, 1702051]","[738980, 580177, 1466260, 1639858, 784889, 182...",1.732709e+09,"[1607695, 1636621, 1407779, 1434110, 992023, 9..."
1,6948,[1535117],"[199732, 1012088, 692593, 88768, 556025, 18895...",1.732531e+09,"[1607695, 1636621, 1407779, 1434110, 992023, 9..."
2,6948,"[1158042, 927325, 1151720, 1682259]","[840370, 1744118, 1910640, 52256, 1276796, 840...",1.732708e+09,"[1607695, 1636621, 1407779, 1434110, 992023, 9..."
3,11094,[1303987],"[1126555, 882946, 1566042, 805346, 1223493, 11...",1.732588e+09,"[1311477, 1483454, 1400183, 847263, 967940, 95..."
4,11897,"[987117, 1929780, 1668032, 1489916, 1016962, 1...","[453067, 1897903, 337651]",1.732699e+09,"[1518787, 1372754, 1613393, 1514777, 1530888, ..."


In [ ]:
# do the same thing for test data
with open('./qilin_rerank/raw_data/recommendation_test.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    test_rec_df = pd.DataFrame(list(reader))
    test_rec_df.columns = test_rec_df.iloc[0]
    test_rec_df = test_rec_df.iloc[1:]

test_rec_result = []
for idx, row in test_rec_df.iterrows():
    user_idx = row['user_idx']
    rec_str = row['rec_result_details_with_idx']
    try:
        rec_list = re.findall(r"\{.*?\}", rec_str, re.DOTALL)
    except Exception as e:
        continue

    gt_note_list = []
    neg_note_list = []
    for rec in rec_list:
        rec_dict = ast.literal_eval(rec)
        if rec_dict.get('click', 0) == 1:
            note_idx = rec_dict.get('note_idx', None)
            timestamp = rec_dict.get('request_timestamp', None)
            gt_note_list.append(note_idx)
        else:
            note_idx = rec_dict.get('note_idx', None)
            timestamp = rec_dict.get('request_timestamp', None)
            neg_note_list.append(note_idx)
    history = row['recent_clicked_note_idxs']
    # change to list
    history = history.strip('[]').split()
    test_rec_result.append([user_idx, gt_note_list, neg_note_list, timestamp, history])

# 转为DataFrame
test_rec_result_df = pd.DataFrame(test_rec_result, columns=['user_idx', 'gt_note_idx', 'neg_note_idx', 'timestamp', 'history'])

# merge the train and test data
rec_result_df = pd.concat([train_rec_result_df, test_rec_result_df])

In [ ]:
# do the same thing for the search data
with open('./qilin_rerank/raw_data/search_train.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    search_df = pd.DataFrame(list(reader))
    search_df.columns = search_df.iloc[0]
    search_df = search_df.iloc[1:]
search_df.head()

,query,query_from_type,recent_clicked_note_idxs,search_idx,session_idx,user_idx,dpr_results,search_result_details_with_idx
1,潜执潜行失忆绘画,1.0,[1702051 958835 1315680 1402517 958835 16104...,38535,18643,6948,"[array([1.81627200e+06, 3.22203461e+02])\n arr...","[{'click': 0.0, 'collect': 0.0, 'comment': 0.0..."
2,渔歌子主旨,1.0,[1151720 927325 1158042 1151720 1535117 9903...,25147,9831,6948,"[array([1.96492600e+06, 3.20901398e+02])\n arr...","[{'click': 1.0, 'collect': 0.0, 'comment': 0.0..."
3,文明校园主题手抄报,3.0,[1702051 958835 1315680 1402517 958835 16104...,39120,13204,6948,"[array([8.12711000e+05, 3.24777679e+02])\n arr...","[{'click': 0.0, 'collect': 0.0, 'comment': 0.0..."
4,文明校园主题手抄报,3.0,[1702051 958835 1315680 1402517 958835 16104...,38372,13204,6948,"[array([8.12711000e+05, 3.24777679e+02])\n arr...","[{'click': 0.0, 'collect': 0.0, 'comment': 0.0..."
5,执潜文潜失忆文,2.0,[1702051 958835 1315680 1402517 958835 16104...,15106,18643,6948,"[array([143810. , 325.84310913])\n a...","[{'click': 1.0, 'collect': 0.0, 'comment': 0.0..."


In [8]:
train_search_result = []
for idx, row in search_df.iterrows():
    user_idx = row['user_idx']
    search_str = row['search_result_details_with_idx']
    search_list = re.findall(r"\{.*?\}", search_str, re.DOTALL)

    gt_note_list = []
    neg_note_list = []
    for search in search_list:
        search = search.replace('nan', 'None') 
        try:
            search_dict = ast.literal_eval(search)
        except Exception as e:
            print(f"解析失败内容：{search}\n错误：{e}")
            continue
        if int(search_dict['click']) == 1:
            note_idx = search_dict.get('note_idx', None)
            timestamp = search_dict.get('search_timestamp', None)
            query = row['query']
            gt_note_list.append(note_idx)
        else:
            note_idx = search_dict.get('note_idx', None)
            timestamp = search_dict.get('search_timestamp', None)
            query = row['query']
            neg_note_list.append(note_idx)
    history = row['recent_clicked_note_idxs']
    # change to list
    history = history.strip('[]').split()
    train_search_result.append([user_idx, gt_note_list, neg_note_list, timestamp, query, history])

train_search_result_df = pd.DataFrame(train_search_result, columns=['user_idx', 'gt_note_idx', 'neg_note_idx', 'timestamp', 'query', 'history'])
train_search_result_df.head()

,user_idx,gt_note_idx,neg_note_idx,timestamp,query,history
0,6948,"[1339821, 1168784]","[1684016, 750617, 1564529, 741794, 208988, 131...",1.732568e+09,潜执潜行失忆绘画,"[1702051, 958835, 1315680, 1402517, 958835, 16..."
1,6948,"[785089, 1322068, 918567, 892353]","[731340, 714294, 438807, 1780677, 572248, 1458...",1.732476e+09,渔歌子主旨,"[1151720, 927325, 1158042, 1151720, 1535117, 9..."
2,6948,[958835],"[173370, 1247113, 1256564, 430205, 334335, 156...",1.732566e+09,文明校园主题手抄报,"[1702051, 958835, 1315680, 1402517, 958835, 16..."
3,6948,[812711],"[497927, 565008, 492294, 1149967, 173370, 5918...",1.732566e+09,文明校园主题手抄报,"[1702051, 958835, 1315680, 1402517, 958835, 16..."
4,6948,"[1211032, 1383331, 927426, 1131213]","[63777, 490775, 503641, 1167665, 680402, 24644...",1.732567e+09,执潜文潜失忆文,"[1702051, 958835, 1315680, 1402517, 958835, 16..."


## 

In [10]:
# add a column 'task' to both rec_result_df and search_result_df
rec_result_df['task'] = 'rec'
search_result_df['task'] = 'search'

# only keep the users and items that have at least 3 interactions for both rec and search
rec_result_df = rec_result_df[rec_result_df['user_idx'].isin(rec_result_df['user_idx'].value_counts()[rec_result_df['user_idx'].value_counts() >= 3].index)]
search_result_df = search_result_df[search_result_df['user_idx'].isin(search_result_df['user_idx'].value_counts()[search_result_df['user_idx'].value_counts() >= 3].index)]

# merge the two dataframes
result_df = pd.concat([rec_result_df, search_result_df])

# sort by timestamp from old to new
result_df = result_df.sort_values(by='timestamp', ascending=True)

# only keep user who has both search and rec behavior
rec_users = set(result_df[result_df['task'] == 'rec']['user_idx'].unique())
search_users = set(result_df[result_df['task'] == 'search']['user_idx'].unique())
both_users = rec_users & search_users
result_df = result_df[result_df['user_idx'].isin(both_users)]


In [11]:
# # split into train/valid/test according to timestamp (7:1.5:1.5)
# train_len = int(len(result_df) * 0.7)
# valid_len = int(len(result_df) * 0.1)
# test_len = len(result_df) - train_len - valid_len

# train_df = result_df.iloc[:train_len]
# valid_df = result_df.iloc[train_len:train_len+valid_len]
# test_df = result_df.iloc[train_len+valid_len:]
# # Remove cold-start users and items from validation and test sets
# valid_df = valid_df[valid_df['user_idx'].isin(train_df['user_idx'].unique())]
# valid_df = valid_df[valid_df['gt_note_idx'].apply(lambda x: any(note in train_df['gt_note_idx'].explode().unique() for note in x))]
# test_df = test_df[test_df['user_idx'].isin(train_df['user_idx'].unique())]
# test_df = test_df[test_df['gt_note_idx'].apply(lambda x: any(note in train_df['gt_note_idx'].explode().unique() for note in x))]

# # split into rec/search according to task
# train_rec_df = train_df[train_df['task'] == 'rec']
# train_search_df = train_df[train_df['task'] == 'search']
# valid_rec_df = valid_df[valid_df['task'] == 'rec']
# valid_search_df = valid_df[valid_df['task'] == 'search']
# test_rec_df = test_df[test_df['task'] == 'rec']
# test_search_df = test_df[test_df['task'] == 'search']


In [12]:
# split into train/valid/test according to leave-one-out strategy
rec_result_df = result_df[result_df['task'] == 'rec']
search_result_df = result_df[result_df['task'] == 'search']
rec_result_df.head()

,user_idx,gt_note_idx,neg_note_idx,timestamp,history,task,query
1,6948,[1535117],"[199732, 1012088, 692593, 88768, 556025, 18895...",1.732531e+09,"[1607695, 1636621, 1407779, 1434110, 992023, 9...",rec,NaN
3,11094,[1303987],"[1126555, 882946, 1566042, 805346, 1223493, 11...",1.732588e+09,"[1311477, 1483454, 1400183, 847263, 967940, 95...",rec,NaN
13,1453,[899931],"[694206, 863727, 1139699]",1.732637e+09,"[798133, 1651368, 1687140, 937627, 1462271, 19...",rec,NaN
24,2292,[1345796],"[1353094, 1090945, 56230]",1.732637e+09,"[1674648, 1491560, 1249581, 1050946, 1570511, ...",rec,NaN
16,12075,[1470719],"[176895, 1526990, 1192189]",1.732637e+09,"[1505186, 875790, 1489946, 1564563, 1390498, 1...",rec,NaN


In [13]:
rec_result_df = result_df[result_df['task'] == 'rec'].copy()
rec_result_df['user_idx'] = rec_result_df['user_idx'].astype(int)

search_result_df = result_df[result_df['task'] == 'search'].copy()
search_result_df['user_idx'] = search_result_df['user_idx'].astype(int)
def split_by_user(df):
    train, valid, test = [], [], []
    for user, user_df in df.groupby('user_idx'):
        user_df = user_df.sort_values('timestamp')
        if len(user_df) >= 3:
            test.append(user_df.iloc[-1])
            valid.append(user_df.iloc[-2])
            train.append(user_df.iloc[:-2])
    train_df = pd.concat(train).reset_index(drop=True) if train else pd.DataFrame(columns=df.columns)
    valid_df = pd.DataFrame(valid).reset_index(drop=True) if valid else pd.DataFrame(columns=df.columns)
    test_df = pd.DataFrame(test).reset_index(drop=True) if test else pd.DataFrame(columns=df.columns)
    return train_df, valid_df, test_df

# 用法
train_rec_df, valid_rec_df, test_rec_df = split_by_user(rec_result_df)
train_search_df, valid_search_df, test_search_df = split_by_user(search_result_df)

train_rec_df.head()
 

,user_idx,gt_note_idx,neg_note_idx,timestamp,history,task,query
0,7,"[1394473, 1474659, 1403838]","[1622156, 1915880, 1632825, 485384, 1828921, 4...",1.732665e+09,"[886649, 830604, 978573, 1635358, 1088587, 115...",rec,NaN
1,7,"[785527, 981107, 1535029, 811066, 815425, 1671...","[289735, 1022359, 1787493, 1614420, 1206033, 5...",1.732680e+09,"[1403838, 1474659, 1394473, 886649, 830604, 97...",rec,NaN
2,7,"[939809, 1019287, 790689]","[1103555, 1146109]",1.732681e+09,"[1205993, 1441828, 1935709, 1511805, 1671203, ...",rec,NaN
3,7,"[853680, 1124052, 1457690, 1653172, 1233997, 1...","[599608, 1221731, 861005, 1876399, 1845948, 21...",1.732686e+09,"[1417784, 851945, 1740300, 1071336, 1358585, 1...",rec,NaN
4,7,"[859676, 1050039, 1202953, 1244848, 1595671, 1...","[119094, 1492005, 1870534, 214379, 980803, 138...",1.732709e+09,"[1565282, 1311207, 917704, 1597522, 1466751, 1...",rec,NaN


In [14]:
train_df = pd.concat([train_rec_df, train_search_df])
valid_df = pd.concat([valid_rec_df, valid_search_df])
test_df = pd.concat([test_rec_df, test_search_df])

In [15]:
# print statistics of the train/valid/test dataframes
print(f"Train data: {len(train_df)}")
print(f"Valid data: {len(valid_df)}")
print(f"Test data: {len(test_df)}")

# for search/rec, also print the data num
print(f"Train rec data: {len(train_rec_df)}")
print(f"Train search data: {len(train_search_df)}")
print(f"Valid rec data: {len(valid_rec_df)}")
print(f"Valid search data: {len(valid_search_df)}")
print(f"Test rec data: {len(test_rec_df)}")
print(f"Test search data: {len(test_search_df)}")

# Calculate unique users and items across all datasets
all_users = pd.concat([train_df, valid_df, test_df])['user_idx'].unique()
all_items = pd.concat([train_df, valid_df, test_df])['gt_note_idx'].explode().unique()

print(f"Total unique users: {len(all_users)}")
print(f"Total unique items: {len(all_items)}")




Train data: 78842
Valid data: 7684
Test data: 7684
Train rec data: 50299
Train search data: 28543
Valid rec data: 3842
Valid search data: 3842
Test rec data: 3842
Test search data: 3842
Total unique users: 3842
Total unique items: 280342


In [ ]:
# save to pkl
train_df.to_pickle('./qilin_rerank/ori_data/train.pkl')
valid_df.to_pickle('./qilin_rerank/ori_data/valid.pkl')
test_df.to_pickle('./qilin_rerank/ori_data/test.pkl')
train_rec_df.to_pickle('./qilin_rerank/ori_data/rec_train.pkl')
train_search_df.to_pickle('./qilin_rerank/ori_data/src_train.pkl')
valid_rec_df.to_pickle('./qilin_rerank/ori_data/rec_valid.pkl')
valid_search_df.to_pickle('./qilin_rerank/ori_data/src_valid.pkl')
test_rec_df.to_pickle('./qilin_rerank/ori_data/rec_test.pkl')
test_search_df.to_pickle('./qilin_rerank/ori_data/src_test.pkl')

In [17]:
train_rec_df.head()

,user_idx,gt_note_idx,neg_note_idx,timestamp,history,task,query
0,7,"[1394473, 1474659, 1403838]","[1622156, 1915880, 1632825, 485384, 1828921, 4...",1.732665e+09,"[886649, 830604, 978573, 1635358, 1088587, 115...",rec,NaN
1,7,"[785527, 981107, 1535029, 811066, 815425, 1671...","[289735, 1022359, 1787493, 1614420, 1206033, 5...",1.732680e+09,"[1403838, 1474659, 1394473, 886649, 830604, 97...",rec,NaN
2,7,"[939809, 1019287, 790689]","[1103555, 1146109]",1.732681e+09,"[1205993, 1441828, 1935709, 1511805, 1671203, ...",rec,NaN
3,7,"[853680, 1124052, 1457690, 1653172, 1233997, 1...","[599608, 1221731, 861005, 1876399, 1845948, 21...",1.732686e+09,"[1417784, 851945, 1740300, 1071336, 1358585, 1...",rec,NaN
4,7,"[859676, 1050039, 1202953, 1244848, 1595671, 1...","[119094, 1492005, 1870534, 214379, 980803, 138...",1.732709e+09,"[1565282, 1311207, 917704, 1597522, 1466751, 1...",rec,NaN
